In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras import Sequential, layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

In [21]:
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=1000)

In [22]:
print(x_train.shape)
print(x_test.shape)

(25000,)
(25000,)


In [23]:
x_train[:2]

array([list([1, 14, 22, 16, 43, 530, 973, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 2, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]),
       list([1, 194, 2, 194, 2, 78, 228, 5, 6, 2, 2, 2, 134, 26, 4, 715, 8, 118, 2, 14, 394, 20, 13, 119, 954, 189,

In [24]:
from keras.utils import pad_sequences

# Adding padding of o 
x_train = pad_sequences(x_train, padding='post', maxlen=100)
x_test = pad_sequences(x_test, padding='post', maxlen=100)

print(x_train.shape)
print()
print(x_train)

(25000, 100)

[[  2  33   6 ...  19 178  32]
 [163  11   2 ...  16 145  95]
 [  2   4   2 ...   7 129 113]
 ...
 [ 11   6   2 ...   4   2   2]
 [100   2   8 ...  12   9  23]
 [ 78   2  17 ... 204 131   9]]


In [25]:
import optuna

In [27]:
def my_model(trial):
    lr = trial.suggest_float('lr', 1e-8, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1,5, step=1)

 
    model = keras.Sequential([
        keras.Input(shape=(100,))    ,
        layers.Embedding(input_dim=1000, output_dim=32)            # The Embedding layer is REQUIRED for text data integer tokens
    ])

    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)
        activation_fn = trial.suggest_categorical(f'activation_fn_{i}', ['relu', 'leaky_relu', 'elu', 'tanh'])
        dropouts = trial.suggest_float(f'dropouts_{i}', 0.0,0.5, step=0.1)
        
        initializer = trial.suggest_categorical(f'initializer_{i}', ['he_normal', 'glorot_normal'])
        regularizer = trial.suggest_categorical(f'regularizer_{i}', ['l1','l2'])
        reg_rate = trial.suggest_float(f'reg_rate_{i}', 1e-8, 1e-2, log=True)
        if regularizer == 'l1':
            chosen_regularizer = regularizers.l1(reg_rate)
        else:
            chosen_regularizer = regularizers.l2(reg_rate)

        bias_initializer = trial.suggest_categorical(f'bias_initializer_{i}', ['he_normal', 'glorot_normal', 'he_uniform', 'glorot_uniform'])
        bias_regularizer = trial.suggest_categorical(f'bias_regularizer_{i}', ['l1','l2'])
        bias_reg_rate = trial.suggest_float(f'bias_reg_rate_{i}', 1e-8, 1e-2, log=True)

        if bias_regularizer == 'l1':
            chosen_bias_regularizer = regularizers.l1(bias_reg_rate)
        else:
            chosen_bias_regularizer = regularizers.l2(bias_reg_rate)
        

        if i < (n_layers -1):
            return_sequence = True
        else:
            return_sequence = False

        model.add(layers.SimpleRNN(
            neurons, 
            return_sequences = return_sequence,
            activation = activation_fn,
            kernel_initializer=initializer, 
            kernel_regularizer=chosen_regularizer, 
            bias_initializer=bias_initializer,
            bias_regularizer=chosen_bias_regularizer            
        ))

        if dropouts > 0.0:
            model.add(layers.Dropout(dropouts))

    neurons = trial.suggest_int(f'ANN_neurons', 1,200, step=16)
    activation_fn = trial.suggest_categorical(f'ANN_activation_fn', ['relu', 'leaky_relu', 'elu', 'tanh'])
    model.add(layers.Dense(
        neurons,
        activation = activation_fn        
    ))
    
    model.add(layers.Dense(1, activation='sigmoid'))


    model.compile(
        loss = keras.losses.BinaryCrossentropy(),
        optimizer = keras.optimizers.Adam(lr),
        metrics = ['accuracy']    
    )
    

    early_stopping = EarlyStopping(
        monitor = 'val_accuracy',
        patience = 3,
        restore_best_weights = True,
        verbose = 1
    )


    history = model.fit(
        x_train, y_train,
        epochs = 5,
        batch_size = 256,
        validation_data = (x_test, y_test),
        callbacks = [early_stopping],        
        verbose = 1
    )

    model.save(f"model_trial_{trial.number}.keras")

    return max(history.history['val_accuracy'])


study = optuna.create_study(direction='maximize')
study.optimize(my_model, n_trials = 10)


[I 2026-09-28 10:05:50,550] A new study created in memory with name: no-name-d244a447-706e-4e40-a4f9-e64229dc6d81


Epoch 1/5


/tmp/ipykernel_58/2048346450.py:12: UserWarning: The distribution is specified by [1, 200] and step=16, but the range is not divisible by `step`. It will be replaced with [1, 193].
  neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)
/tmp/ipykernel_58/2048346450.py:52: UserWarning: The distribution is specified by [1, 200] and step=16, but the range is not divisible by `step`. It will be replaced with [1, 193].
  neurons = trial.suggest_int(f'ANN_neurons', 1,200, step=16)


 3/98 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.5117 - loss: 1.1797

I0000 00:00:1790589959.095443     210 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

[I 2026-09-28 10:09:53,530] Trial 7 finished with value: 0.5 and parameters: {'lr': 4.823218194589542e-08, 'n_layers': 5, 'neurons_0': 193, 'activation_fn_0': 'elu', 'dropouts_0': 0.4, 'initializer_0': 'glorot_normal', 'regularizer_0': 'l2', 'reg_rate_0': 6.000483759455138e-07, 'bias_initializer_0': 'glorot_uniform', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 1.1293055548770848e-06, 'neurons_1': 193, 'activation_fn_1': 'tanh', 'dropouts_1': 0.0, 'initializer_1': 'glorot_normal', 'regularizer_1': 'l1', 'reg_rate_1': 7.778491312204725e-07, 'bias_initializer_1': 'he_normal', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 2.9479489167232833e-06, 'neurons_2': 193, 'activation_fn_2': 'tanh', 'dropouts_2': 0.0, 'initializer_2': 'he_normal', 'regularizer_2': 'l1', 'reg_rate_2': 2.5674076699096277e-08, 'bias_initializer_2': 'he_uniform', 'bias_regularizer_2': 'l1', 'bias_reg_rate_2': 0.006030459662330657, 'neurons_3': 113, 'activation_fn_3': 'relu', 'dropouts_3': 0.0, 'initializer_3': 'glor

Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 12s 68ms/step - accuracy: 0.4964 - loss: 0.7222 - val_accuracy: 0.5060 - val_loss: 0.6940
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.4971 - loss: 0.7167 - val_accuracy: 0.5018 - val_loss: 0.6944
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5004 - loss: 0.7127 - val_accuracy: 0.4997 - val_loss: 0.6946
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5074 - loss: 0.7074 - val_accuracy: 0.5002 - val_loss: 0.6947
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


[I 2026-09-28 10:10:12,241] Trial 8 finished with value: 0.5059999823570251 and parameters: {'lr': 1.6363703050661866e-05, 'n_layers': 2, 'neurons_0': 17, 'activation_fn_0': 'leaky_relu', 'dropouts_0': 0.5, 'initializer_0': 'he_normal', 'regularizer_0': 'l1', 'reg_rate_0': 8.809336996617328e-06, 'bias_initializer_0': 'he_uniform', 'bias_regularizer_0': 'l2', 'bias_reg_rate_0': 9.075410090832277e-07, 'neurons_1': 33, 'activation_fn_1': 'elu', 'dropouts_1': 0.1, 'initializer_1': 'glorot_normal', 'regularizer_1': 'l2', 'reg_rate_1': 2.2283908493817138e-08, 'bias_initializer_1': 'glorot_uniform', 'bias_regularizer_1': 'l2', 'bias_reg_rate_1': 2.13741103276218e-07, 'ANN_neurons': 33, 'ANN_activation_fn': 'leaky_relu'}. Best is trial 5 with value: 0.518559992313385.


Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 166ms/step - accuracy: 0.5045 - loss: 6.9556 - val_accuracy: 0.4996 - val_loss: 6.7776
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.5059 - loss: 6.7133 - val_accuracy: 0.4991 - val_loss: 6.5615
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.4999 - loss: 6.4978 - val_accuracy: 0.5002 - val_loss: 6.3549
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.4986 - loss: 6.2869 - val_accuracy: 0.5006 - val_loss: 6.1532
Epoch 5/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.5019 - loss: 6.0778 - val_accuracy: 0.5060 - val_loss: 5.9533
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-28 10:11:08,650] Trial 9 finished with value: 0.5059999823570251 and parameters: {'lr': 3.2891276527533525e-05, 'n_layers': 5, 'neurons_0': 177, 'activation_fn_0': 'elu', 'dropouts_0': 0.4, 'initializer_0': 'glorot_normal', 'regularizer_0': 'l1', 'reg_rate_0': 0.0019061778037853985, 'bias_initializer_0': 'he_uniform', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 0.00013662322499512536, 'neurons_1': 177, 'activation_fn_1': 'leaky_relu', 'dropouts_1': 0.2, 'initializer_1': 'he_normal', 'regularizer_1': 'l1', 'reg_rate_1': 1.0783133497605322e-06, 'bias_initializer_1': 'he_normal', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 9.589588587914122e-05, 'neurons_2': 97, 'activation_fn_2': 'leaky_relu', 'dropouts_2': 0.2, 'initializer_2': 'he_normal', 'regularizer_2': 'l2', 'reg_rate_2': 0.0006143702350338715, 'bias_initializer_2': 'glorot_normal', 'bias_regularizer_2': 'l2', 'bias_reg_rate_2': 1.1249422825400659e-05, 'neurons_3': 113, 'activation_fn_3': 'leaky_relu', 'dropouts_3'

In [29]:
print(study.best_trial.number)
print()
study.best_params

5



{'lr': 1.6352694095992686e-05,
 'n_layers': 1,
 'neurons_0': 193,
 'activation_fn_0': 'leaky_relu',
 'dropouts_0': 0.5,
 'initializer_0': 'he_normal',
 'regularizer_0': 'l2',
 'reg_rate_0': 0.0007377832743045447,
 'bias_initializer_0': 'glorot_normal',
 'bias_regularizer_0': 'l1',
 'bias_reg_rate_0': 1.7253531582464675e-06,
 'ANN_neurons': 193,
 'ANN_activation_fn': 'tanh'}

In [30]:
num = study.best_trial.number
model = keras.models.load_model(f"model_trial_{num}.keras")
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 100, 32)        │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_17 (SimpleRNN)       │ (None, 193)            │        43,618 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 193)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 193)            │        37,442 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │           194 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 339,764 (1.30 MB)

 Trainable params: 113,254 (442.40 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 226,510 (884.81 KB)

In [38]:
model.fit(
    x_train, y_train,
    epochs = 100,
    batch_size=256
)

Epoch 1/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9692 - loss: 0.2527
Epoch 2/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9631 - loss: 0.2639
Epoch 3/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9698 - loss: 0.2520
Epoch 4/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9695 - loss: 0.2533
Epoch 5/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9691 - loss: 0.2505
Epoch 6/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9661 - loss: 0.2576
Epoch 7/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9696 - loss: 0.2481
Epoch 8/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9708 - loss: 0.2470
Epoch 9/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9709 - loss: 0.2463
Epoch 10/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9682 - loss: 0.2504
Epoch 11/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9660 - loss: 0.2548
Epoch 12/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step

In [39]:
pred = model.predict(x_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step


In [ ]:
# 1. Predict probabilities
pred = model.predict(x_test)

# 2. Convert probabilities into binary classes (0 or 1) using a 0.5 threshold
predicted_classes = (pred >= 0.5).astype(int).flatten()

# 3. Calculate true accuracy
accuracy = np.mean(predicted_classes == y_test)
print(f"Corrected Test Accuracy: {accuracy * 100:.2f}%\n")

# 4. Generate proper classification metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(y_test, predicted_classes))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predicted_classes))